In [ ]:
import pandas as pd

ds = pd.read_csv('functions_for_django_with_cwe.csv')


In [ ]:
ds.info()

In [ ]:
ds.to_csv('functions_for_django_with_cwe',index=False,quoting=1)

In [ ]:
df=pd.read_csv('ground_truth_functions_for_ansible.csv')
df.info()

In [ ]:
df = df.dropna(subset=['vulnerable_function_source'])

df = df[df['vulnerable_function_source'].str.strip() != '']
print(df.shape)

In [ ]:
df.to_csv('functions_for_ansible.csv',index=False,quoting=1)

In [ ]:

import re
import subprocess
import json
import os
import sys

# Check for pandas installation
try:
    import pandas as pd
except ImportError:
    print("Pandas is not installed. Install it using 'pip install pandas'.")
    sys.exit(1)

# Parse command-line arguments for input and output file paths

input_csv = "functions_for_ansible.csv"
output_csv = "functions_for_ansible_with_cwe.csv"

# Load the dataset
try:
    df = pd.read_csv(input_csv)
except Exception as e:
    print(f"Error reading CSV file: {e}")
    sys.exit(1)

if 'vulnerable_function_source' not in df.columns:
    print("Input CSV must contain a 'vulnerable_function_source' column.")
    sys.exit(1)

# Regex patterns mapping common vulnerability patterns to CWE IDs (fallback heuristic)
regex_patterns = {
    # CWE-78: OS Command Injection
    "CWE-78": [
        re.compile(r"os\.system\(", re.DOTALL),
        re.compile(r"os\.popen\(", re.DOTALL),
        re.compile(r"subprocess\.(?:run|Popen|call|check_output)\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)Popen\([^)]*shell\s*=\s*True", re.DOTALL),  # unqualified Popen (if imported)
        re.compile(r"(?<!\.)run\([^)]*shell\s*=\s*True", re.DOTALL),    # unqualified run (if imported)
        re.compile(r"(?<!\.)call\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)check_output\([^)]*shell\s*=\s*True", re.DOTALL),
    ],
    # CWE-89: SQL Injection
    "CWE-89": [
        re.compile(r"\.execute\([^)]*(SELECT|INSERT|UPDATE|DELETE)", re.IGNORECASE | re.DOTALL),
        re.compile(r"\.execute\([^)]*\+[^)]*\)", re.DOTALL),       # string concatenation in query
        re.compile(r"\.execute\([^)]*\.format\(", re.DOTALL),      # use of format() in query
        re.compile(r"\.execute\([^)]*f[\'\"]", re.DOTALL),         # f-string in query
        re.compile(r"\.execute\([^)]*%[^)]*\)", re.DOTALL),        # %-formatting in query
    ],
    # CWE-79: Cross-Site Scripting (XSS)
    "CWE-79": [
        re.compile(r"mark_safe\(", re.DOTALL),
        re.compile(r"jinja2\.Environment\([^)]*autoescape\s*=\s*False", re.DOTALL),
        re.compile(r"autoescape\s*=\s*False", re.DOTALL),  # any occurrence of disabling autoescape
        re.compile(r"mako\.template", re.DOTALL),
        re.compile(r"mako\.Template", re.DOTALL),
    ],
    # CWE-94: Code Injection (Eval/Exec)
    "CWE-94": [
        re.compile(r"\bexec\s*\(", re.DOTALL),
        re.compile(r"\beval\s*\(", re.DOTALL),
    ],
    # CWE-259: Hardcoded Password
    "CWE-259": [
        re.compile(r"password\s*=\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
        re.compile(r"['\"]password['\"]\s*:\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
    ],
    # CWE-732: Insecure File Permissions
    "CWE-732": [
        re.compile(r"os\.chmod\([^,]+,\s*(?:0o777|777|0x1ff|511)\b", re.DOTALL),
    ],
    # CWE-605: Binding to All Interfaces
    "CWE-605": [
        re.compile(r"0\.0\.0\.0"),
    ],
    # CWE-295: Improper Certificate Validation
    "CWE-295": [
        re.compile(r"requests\.(?:get|post|put|delete|head|options)\([^)]*verify\s*=\s*False", re.DOTALL),
        re.compile(r"ssl\._create_unverified_context\(", re.DOTALL),
        re.compile(r"set_missing_host_key_policy\([^)]*AutoAddPolicy", re.DOTALL),
    ],
    # CWE-330: Use of Insufficiently Random Values
    "CWE-330": [
        re.compile(r"random\.(?:Random|random|randint|randrange|choice|choices|uniform|triangular|sample|getrandbits|randbytes)\(", re.DOTALL),
    ],
    # CWE-502: Deserialization of Untrusted Data
    "CWE-502": [
        re.compile(r"pickle\.load(s|s)?\(", re.DOTALL),
        re.compile(r"dill\.load(s|s)?\(", re.DOTALL),
        re.compile(r"shelve\.open\(", re.DOTALL),
        re.compile(r"marshal\.load(s|s)?\(", re.DOTALL),
        re.compile(r"jsonpickle\.decode\(", re.DOTALL),
    ],
    # CWE-20: Improper Input Validation (e.g., unsafe YAML load)
    "CWE-20": [
        re.compile(r"yaml\.load\(", re.DOTALL),
    ],
    # CWE-22: Path Traversal (unsafe file extraction)
    "CWE-22": [
        re.compile(r"tarfile\.extractall\(", re.DOTALL),
        re.compile(r"tarfile\.extract\(", re.DOTALL),
        re.compile(r"ZipFile\.extractall\(", re.DOTALL),
        re.compile(r"ZipFile\.extract\(", re.DOTALL),
        re.compile(r"extractall\(", re.DOTALL),  # covers zipfile objects
    ],
    # CWE-703: Improper Error Handling
    "CWE-703": [
        re.compile(r"except\s*:\s*pass", re.DOTALL),
        re.compile(r"except\s*:\s*continue", re.DOTALL),
        re.compile(r"\bassert\s+", re.DOTALL),
    ],
    # CWE-327: Use of Broken or Risky Cryptographic Algorithm
    "CWE-327": [
        re.compile(r"hashlib\.(?:md5|sha1)\(", re.DOTALL),
        re.compile(r"hashlib\.new\(\s*['\"](?:MD5|md5|SHA1|sha1)['\"]", re.DOTALL),
        re.compile(r"(?:\b|\.)?(?:ARC2|ARC4|Blowfish|DES|XOR|CAST5|IDEA|SEED|TripleDES)\.new\(", re.DOTALL),
        re.compile(r"ssl\.PROTOCOL_(?:SSLv2|SSLv3|TLSv1)\b"),
    ],
}

predicted_cwes = []
temp_file = "__bandit_temp_code.py"

for code in df['vulnerable_function_source']:
    code_str = "" if pd.isna(code) else str(code)
    if code_str.strip() == "":
        predicted_cwes.append("")  # empty snippet
        continue

    # Write code snippet to a temp file for Bandit analysis
    try:
        with open(temp_file, "w", encoding="utf-8") as f:
            f.write(code_str)
    except Exception as e:
        # If writing fails, skip Bandit and use regex
        bandit_cwe_list = []
    else:
        # Run Bandit security scanner on the snippet
        bandit_cwe_list = []
        try:
            result = subprocess.run(
                ["bandit", "-f", "json", "-q", temp_file],
                capture_output=True, text=True, timeout=10
            )
            # If Bandit ran successfully, parse JSON output
            if result.returncode == 0 and result.stdout:
                data = json.loads(result.stdout)
                if data.get("results"):
                    for issue in data["results"]:
                        cwe_info = issue.get("issue_cwe")
                        if cwe_info:
                            cwe_id = cwe_info.get("id")
                            if cwe_id:
                                bandit_cwe_list.append(f"CWE-{cwe_id}")
                # If Bandit reported errors and no results, treat as no CWE found
                if data.get("errors") and not bandit_cwe_list:
                    bandit_cwe_list = []
            else:
                # Non-zero return code or no output -> assume Bandit did not find issues or failed
                bandit_cwe_list = []
        except Exception:
            # If Bandit is not installed or execution failed, fallback to regex
            bandit_cwe_list = []

    # Remove duplicate CWEs from Bandit results
    bandit_cwe_list = sorted(set(bandit_cwe_list), key=lambda x: int(x.split('-')[1])) if bandit_cwe_list else []

    if bandit_cwe_list:
        # Use CWE IDs reported by Bandit
        cwe_ids = bandit_cwe_list
    else:
        # Fallback: use regex patterns to infer CWE IDs
        found_cwe_set = set()
        for cwe, patterns in regex_patterns.items():
            for pattern in patterns:
                if pattern.search(code_str):
                    found_cwe_set.add(cwe)
                    break  # one match is enough for this CWE
        cwe_ids = sorted(found_cwe_set, key=lambda x: int(x.split('-')[1])) if found_cwe_set else []

    # Format the list of CWE IDs as a comma-separated string (or empty)
    predicted_cwes.append(", ".join(cwe_ids) if cwe_ids else "")

# Cleanup temp file
if os.path.exists(temp_file):
    os.remove(temp_file)

# Add the results to DataFrame and save to output CSV
df['predicted_cwe_ids'] = predicted_cwes
try:
    df.to_csv(output_csv, index=False)
    print(f"Predicted CWE IDs saved to '{output_csv}'.")
except Exception as e:
    print(f"Error writing output CSV: {e}")


In [ ]:
df['predicted_cwe_ids'].value_counts()

In [ ]:
# Replace any empty strings (or NaNs) in the column with “CWE-Unknown”
df['predicted_cwe_ids'] = df['predicted_cwe_ids'].fillna('').replace('', 'CWE-Unknown')


In [ ]:
df.to_csv('functions_for_django_with_cwe.csv',index=False,quoting=1)

In [ ]:
import pandas as pd

df = pd.read_csv('ground_truth_functions_for_w3af.csv')
df = df.dropna(subset=['vulnerable_function_source'])

df = df[df['vulnerable_function_source'].str.strip() != '']
print(df.shape)
df.info()

In [ ]:
df.to_csv('functions_for_w3af.csv',index= False,quoting =1)

In [ ]:

import re
import subprocess
import json
import os
import sys

# Check for pandas installation
try:
    import pandas as pd
except ImportError:
    print("Pandas is not installed. Install it using 'pip install pandas'.")
    sys.exit(1)

# Parse command-line arguments for input and output file paths

input_csv = "functions_for_w3af.csv"
output_csv = "functions_for_w3af_with_cwe.csv"

# Load the dataset
try:
    df = pd.read_csv(input_csv)
except Exception as e:
    print(f"Error reading CSV file: {e}")
    sys.exit(1)

if 'vulnerable_function_source' not in df.columns:
    print("Input CSV must contain a 'vulnerable_function_source' column.")
    sys.exit(1)

# Regex patterns mapping common vulnerability patterns to CWE IDs (fallback heuristic)
regex_patterns = {
    # CWE-78: OS Command Injection
    "CWE-78": [
        re.compile(r"os\.system\(", re.DOTALL),
        re.compile(r"os\.popen\(", re.DOTALL),
        re.compile(r"subprocess\.(?:run|Popen|call|check_output)\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)Popen\([^)]*shell\s*=\s*True", re.DOTALL),  # unqualified Popen (if imported)
        re.compile(r"(?<!\.)run\([^)]*shell\s*=\s*True", re.DOTALL),    # unqualified run (if imported)
        re.compile(r"(?<!\.)call\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)check_output\([^)]*shell\s*=\s*True", re.DOTALL),
    ],
    # CWE-89: SQL Injection
    "CWE-89": [
        re.compile(r"\.execute\([^)]*(SELECT|INSERT|UPDATE|DELETE)", re.IGNORECASE | re.DOTALL),
        re.compile(r"\.execute\([^)]*\+[^)]*\)", re.DOTALL),       # string concatenation in query
        re.compile(r"\.execute\([^)]*\.format\(", re.DOTALL),      # use of format() in query
        re.compile(r"\.execute\([^)]*f[\'\"]", re.DOTALL),         # f-string in query
        re.compile(r"\.execute\([^)]*%[^)]*\)", re.DOTALL),        # %-formatting in query
    ],
    # CWE-79: Cross-Site Scripting (XSS)
    "CWE-79": [
        re.compile(r"mark_safe\(", re.DOTALL),
        re.compile(r"jinja2\.Environment\([^)]*autoescape\s*=\s*False", re.DOTALL),
        re.compile(r"autoescape\s*=\s*False", re.DOTALL),  # any occurrence of disabling autoescape
        re.compile(r"mako\.template", re.DOTALL),
        re.compile(r"mako\.Template", re.DOTALL),
    ],
    # CWE-94: Code Injection (Eval/Exec)
    "CWE-94": [
        re.compile(r"\bexec\s*\(", re.DOTALL),
        re.compile(r"\beval\s*\(", re.DOTALL),
    ],
    # CWE-259: Hardcoded Password
    "CWE-259": [
        re.compile(r"password\s*=\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
        re.compile(r"['\"]password['\"]\s*:\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
    ],
    # CWE-732: Insecure File Permissions
    "CWE-732": [
        re.compile(r"os\.chmod\([^,]+,\s*(?:0o777|777|0x1ff|511)\b", re.DOTALL),
    ],
    # CWE-605: Binding to All Interfaces
    "CWE-605": [
        re.compile(r"0\.0\.0\.0"),
    ],
    # CWE-295: Improper Certificate Validation
    "CWE-295": [
        re.compile(r"requests\.(?:get|post|put|delete|head|options)\([^)]*verify\s*=\s*False", re.DOTALL),
        re.compile(r"ssl\._create_unverified_context\(", re.DOTALL),
        re.compile(r"set_missing_host_key_policy\([^)]*AutoAddPolicy", re.DOTALL),
    ],
    # CWE-330: Use of Insufficiently Random Values
    "CWE-330": [
        re.compile(r"random\.(?:Random|random|randint|randrange|choice|choices|uniform|triangular|sample|getrandbits|randbytes)\(", re.DOTALL),
    ],
    # CWE-502: Deserialization of Untrusted Data
    "CWE-502": [
        re.compile(r"pickle\.load(s|s)?\(", re.DOTALL),
        re.compile(r"dill\.load(s|s)?\(", re.DOTALL),
        re.compile(r"shelve\.open\(", re.DOTALL),
        re.compile(r"marshal\.load(s|s)?\(", re.DOTALL),
        re.compile(r"jsonpickle\.decode\(", re.DOTALL),
    ],
    # CWE-20: Improper Input Validation (e.g., unsafe YAML load)
    "CWE-20": [
        re.compile(r"yaml\.load\(", re.DOTALL),
    ],
    # CWE-22: Path Traversal (unsafe file extraction)
    "CWE-22": [
        re.compile(r"tarfile\.extractall\(", re.DOTALL),
        re.compile(r"tarfile\.extract\(", re.DOTALL),
        re.compile(r"ZipFile\.extractall\(", re.DOTALL),
        re.compile(r"ZipFile\.extract\(", re.DOTALL),
        re.compile(r"extractall\(", re.DOTALL),  # covers zipfile objects
    ],
    # CWE-703: Improper Error Handling
    "CWE-703": [
        re.compile(r"except\s*:\s*pass", re.DOTALL),
        re.compile(r"except\s*:\s*continue", re.DOTALL),
        re.compile(r"\bassert\s+", re.DOTALL),
    ],
    # CWE-327: Use of Broken or Risky Cryptographic Algorithm
    "CWE-327": [
        re.compile(r"hashlib\.(?:md5|sha1)\(", re.DOTALL),
        re.compile(r"hashlib\.new\(\s*['\"](?:MD5|md5|SHA1|sha1)['\"]", re.DOTALL),
        re.compile(r"(?:\b|\.)?(?:ARC2|ARC4|Blowfish|DES|XOR|CAST5|IDEA|SEED|TripleDES)\.new\(", re.DOTALL),
        re.compile(r"ssl\.PROTOCOL_(?:SSLv2|SSLv3|TLSv1)\b"),
    ],
}

predicted_cwes = []
temp_file = "__bandit_temp_code.py"

for code in df['vulnerable_function_source']:
    code_str = "" if pd.isna(code) else str(code)
    if code_str.strip() == "":
        predicted_cwes.append("")  # empty snippet
        continue

    # Write code snippet to a temp file for Bandit analysis
    try:
        with open(temp_file, "w", encoding="utf-8") as f:
            f.write(code_str)
    except Exception as e:
        # If writing fails, skip Bandit and use regex
        bandit_cwe_list = []
    else:
        # Run Bandit security scanner on the snippet
        bandit_cwe_list = []
        try:
            result = subprocess.run(
                ["bandit", "-f", "json", "-q", temp_file],
                capture_output=True, text=True, timeout=10
            )
            # If Bandit ran successfully, parse JSON output
            if result.returncode == 0 and result.stdout:
                data = json.loads(result.stdout)
                if data.get("results"):
                    for issue in data["results"]:
                        cwe_info = issue.get("issue_cwe")
                        if cwe_info:
                            cwe_id = cwe_info.get("id")
                            if cwe_id:
                                bandit_cwe_list.append(f"CWE-{cwe_id}")
                # If Bandit reported errors and no results, treat as no CWE found
                if data.get("errors") and not bandit_cwe_list:
                    bandit_cwe_list = []
            else:
                # Non-zero return code or no output -> assume Bandit did not find issues or failed
                bandit_cwe_list = []
        except Exception:
            # If Bandit is not installed or execution failed, fallback to regex
            bandit_cwe_list = []

    # Remove duplicate CWEs from Bandit results
    bandit_cwe_list = sorted(set(bandit_cwe_list), key=lambda x: int(x.split('-')[1])) if bandit_cwe_list else []

    if bandit_cwe_list:
        # Use CWE IDs reported by Bandit
        cwe_ids = bandit_cwe_list
    else:
        # Fallback: use regex patterns to infer CWE IDs
        found_cwe_set = set()
        for cwe, patterns in regex_patterns.items():
            for pattern in patterns:
                if pattern.search(code_str):
                    found_cwe_set.add(cwe)
                    break  # one match is enough for this CWE
        cwe_ids = sorted(found_cwe_set, key=lambda x: int(x.split('-')[1])) if found_cwe_set else []

    # Format the list of CWE IDs as a comma-separated string (or empty)
    predicted_cwes.append(", ".join(cwe_ids) if cwe_ids else "")

# Cleanup temp file
if os.path.exists(temp_file):
    os.remove(temp_file)

# Add the results to DataFrame and save to output CSV
df['predicted_cwe_ids'] = predicted_cwes
try:
    df.to_csv(output_csv, index=False)
    print(f"Predicted CWE IDs saved to '{output_csv}'.")
except Exception as e:
    print(f"Error writing output CSV: {e}")


In [ ]:
df['predicted_cwe_ids'].value_counts()

In [ ]:
import pandas as pd

df = pd.read_csv('ground_truth_functions_for_flask.csv')
df = df.dropna(subset=['vulnerable_function_source'])

df = df[df['vulnerable_function_source'].str.strip() != '']
print(df.shape)
df.info()

In [ ]:
df.to_csv('functions_for_flask.csv',index= False,quoting =1)

In [ ]:

import re
import subprocess
import json
import os
import sys

# Check for pandas installation
try:
    import pandas as pd
except ImportError:
    print("Pandas is not installed. Install it using 'pip install pandas'.")
    sys.exit(1)

# Parse command-line arguments for input and output file paths

input_csv = "functions_for_flask.csv"
output_csv = "functions_for_flask_with_cwe.csv"

# Load the dataset
try:
    df = pd.read_csv(input_csv)
except Exception as e:
    print(f"Error reading CSV file: {e}")
    sys.exit(1)

if 'vulnerable_function_source' not in df.columns:
    print("Input CSV must contain a 'vulnerable_function_source' column.")
    sys.exit(1)

# Regex patterns mapping common vulnerability patterns to CWE IDs (fallback heuristic)
regex_patterns = {
    # CWE-78: OS Command Injection
    "CWE-78": [
        re.compile(r"os\.system\(", re.DOTALL),
        re.compile(r"os\.popen\(", re.DOTALL),
        re.compile(r"subprocess\.(?:run|Popen|call|check_output)\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)Popen\([^)]*shell\s*=\s*True", re.DOTALL),  # unqualified Popen (if imported)
        re.compile(r"(?<!\.)run\([^)]*shell\s*=\s*True", re.DOTALL),    # unqualified run (if imported)
        re.compile(r"(?<!\.)call\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)check_output\([^)]*shell\s*=\s*True", re.DOTALL),
    ],
    # CWE-89: SQL Injection
    "CWE-89": [
        re.compile(r"\.execute\([^)]*(SELECT|INSERT|UPDATE|DELETE)", re.IGNORECASE | re.DOTALL),
        re.compile(r"\.execute\([^)]*\+[^)]*\)", re.DOTALL),       # string concatenation in query
        re.compile(r"\.execute\([^)]*\.format\(", re.DOTALL),      # use of format() in query
        re.compile(r"\.execute\([^)]*f[\'\"]", re.DOTALL),         # f-string in query
        re.compile(r"\.execute\([^)]*%[^)]*\)", re.DOTALL),        # %-formatting in query
    ],
    # CWE-79: Cross-Site Scripting (XSS)
    "CWE-79": [
        re.compile(r"mark_safe\(", re.DOTALL),
        re.compile(r"jinja2\.Environment\([^)]*autoescape\s*=\s*False", re.DOTALL),
        re.compile(r"autoescape\s*=\s*False", re.DOTALL),  # any occurrence of disabling autoescape
        re.compile(r"mako\.template", re.DOTALL),
        re.compile(r"mako\.Template", re.DOTALL),
    ],
    # CWE-94: Code Injection (Eval/Exec)
    "CWE-94": [
        re.compile(r"\bexec\s*\(", re.DOTALL),
        re.compile(r"\beval\s*\(", re.DOTALL),
    ],
    # CWE-259: Hardcoded Password
    "CWE-259": [
        re.compile(r"password\s*=\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
        re.compile(r"['\"]password['\"]\s*:\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
    ],
    # CWE-732: Insecure File Permissions
    "CWE-732": [
        re.compile(r"os\.chmod\([^,]+,\s*(?:0o777|777|0x1ff|511)\b", re.DOTALL),
    ],
    # CWE-605: Binding to All Interfaces
    "CWE-605": [
        re.compile(r"0\.0\.0\.0"),
    ],
    # CWE-295: Improper Certificate Validation
    "CWE-295": [
        re.compile(r"requests\.(?:get|post|put|delete|head|options)\([^)]*verify\s*=\s*False", re.DOTALL),
        re.compile(r"ssl\._create_unverified_context\(", re.DOTALL),
        re.compile(r"set_missing_host_key_policy\([^)]*AutoAddPolicy", re.DOTALL),
    ],
    # CWE-330: Use of Insufficiently Random Values
    "CWE-330": [
        re.compile(r"random\.(?:Random|random|randint|randrange|choice|choices|uniform|triangular|sample|getrandbits|randbytes)\(", re.DOTALL),
    ],
    # CWE-502: Deserialization of Untrusted Data
    "CWE-502": [
        re.compile(r"pickle\.load(s|s)?\(", re.DOTALL),
        re.compile(r"dill\.load(s|s)?\(", re.DOTALL),
        re.compile(r"shelve\.open\(", re.DOTALL),
        re.compile(r"marshal\.load(s|s)?\(", re.DOTALL),
        re.compile(r"jsonpickle\.decode\(", re.DOTALL),
    ],
    # CWE-20: Improper Input Validation (e.g., unsafe YAML load)
    "CWE-20": [
        re.compile(r"yaml\.load\(", re.DOTALL),
    ],
    # CWE-22: Path Traversal (unsafe file extraction)
    "CWE-22": [
        re.compile(r"tarfile\.extractall\(", re.DOTALL),
        re.compile(r"tarfile\.extract\(", re.DOTALL),
        re.compile(r"ZipFile\.extractall\(", re.DOTALL),
        re.compile(r"ZipFile\.extract\(", re.DOTALL),
        re.compile(r"extractall\(", re.DOTALL),  # covers zipfile objects
    ],
    # CWE-703: Improper Error Handling
    "CWE-703": [
        re.compile(r"except\s*:\s*pass", re.DOTALL),
        re.compile(r"except\s*:\s*continue", re.DOTALL),
        re.compile(r"\bassert\s+", re.DOTALL),
    ],
    # CWE-327: Use of Broken or Risky Cryptographic Algorithm
    "CWE-327": [
        re.compile(r"hashlib\.(?:md5|sha1)\(", re.DOTALL),
        re.compile(r"hashlib\.new\(\s*['\"](?:MD5|md5|SHA1|sha1)['\"]", re.DOTALL),
        re.compile(r"(?:\b|\.)?(?:ARC2|ARC4|Blowfish|DES|XOR|CAST5|IDEA|SEED|TripleDES)\.new\(", re.DOTALL),
        re.compile(r"ssl\.PROTOCOL_(?:SSLv2|SSLv3|TLSv1)\b"),
    ],
}

predicted_cwes = []
temp_file = "__bandit_temp_code.py"

for code in df['vulnerable_function_source']:
    code_str = "" if pd.isna(code) else str(code)
    if code_str.strip() == "":
        predicted_cwes.append("")  # empty snippet
        continue

    # Write code snippet to a temp file for Bandit analysis
    try:
        with open(temp_file, "w", encoding="utf-8") as f:
            f.write(code_str)
    except Exception as e:
        # If writing fails, skip Bandit and use regex
        bandit_cwe_list = []
    else:
        # Run Bandit security scanner on the snippet
        bandit_cwe_list = []
        try:
            result = subprocess.run(
                ["bandit", "-f", "json", "-q", temp_file],
                capture_output=True, text=True, timeout=10
            )
            # If Bandit ran successfully, parse JSON output
            if result.returncode == 0 and result.stdout:
                data = json.loads(result.stdout)
                if data.get("results"):
                    for issue in data["results"]:
                        cwe_info = issue.get("issue_cwe")
                        if cwe_info:
                            cwe_id = cwe_info.get("id")
                            if cwe_id:
                                bandit_cwe_list.append(f"CWE-{cwe_id}")
                # If Bandit reported errors and no results, treat as no CWE found
                if data.get("errors") and not bandit_cwe_list:
                    bandit_cwe_list = []
            else:
                # Non-zero return code or no output -> assume Bandit did not find issues or failed
                bandit_cwe_list = []
        except Exception:
            # If Bandit is not installed or execution failed, fallback to regex
            bandit_cwe_list = []

    # Remove duplicate CWEs from Bandit results
    bandit_cwe_list = sorted(set(bandit_cwe_list), key=lambda x: int(x.split('-')[1])) if bandit_cwe_list else []

    if bandit_cwe_list:
        # Use CWE IDs reported by Bandit
        cwe_ids = bandit_cwe_list
    else:
        # Fallback: use regex patterns to infer CWE IDs
        found_cwe_set = set()
        for cwe, patterns in regex_patterns.items():
            for pattern in patterns:
                if pattern.search(code_str):
                    found_cwe_set.add(cwe)
                    break  # one match is enough for this CWE
        cwe_ids = sorted(found_cwe_set, key=lambda x: int(x.split('-')[1])) if found_cwe_set else []

    # Format the list of CWE IDs as a comma-separated string (or empty)
    predicted_cwes.append(", ".join(cwe_ids) if cwe_ids else "")

# Cleanup temp file
if os.path.exists(temp_file):
    os.remove(temp_file)

# Add the results to DataFrame and save to output CSV
df['predicted_cwe_ids'] = predicted_cwes
try:
    df.to_csv(output_csv, index=False)
    print(f"Predicted CWE IDs saved to '{output_csv}'.")
except Exception as e:
    print(f"Error writing output CSV: {e}")


In [ ]:
df['predicted_cwe_ids'].value_counts()

In [ ]:
import pandas as pd

df = pd.read_csv('ground_truth_functions_for_paramiko.csv')
df = df.dropna(subset=['vulnerable_function_source'])

df = df[df['vulnerable_function_source'].str.strip() != '']
print(df.shape)
df.info()

In [ ]:
df.to_csv('functions_for_paramiko.csv',index= False,quoting =1)

In [ ]:

import re
import subprocess
import json
import os
import sys

# Check for pandas installation
try:
    import pandas as pd
except ImportError:
    print("Pandas is not installed. Install it using 'pip install pandas'.")
    sys.exit(1)

# Parse command-line arguments for input and output file paths

input_csv = "functions_for_paramiko.csv"
output_csv = "functions_for_paramiko_with_cwe.csv"

# Load the dataset
try:
    df = pd.read_csv(input_csv)
except Exception as e:
    print(f"Error reading CSV file: {e}")
    sys.exit(1)

if 'vulnerable_function_source' not in df.columns:
    print("Input CSV must contain a 'vulnerable_function_source' column.")
    sys.exit(1)

# Regex patterns mapping common vulnerability patterns to CWE IDs (fallback heuristic)
regex_patterns = {
    # CWE-78: OS Command Injection
    "CWE-78": [
        re.compile(r"os\.system\(", re.DOTALL),
        re.compile(r"os\.popen\(", re.DOTALL),
        re.compile(r"subprocess\.(?:run|Popen|call|check_output)\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)Popen\([^)]*shell\s*=\s*True", re.DOTALL),  # unqualified Popen (if imported)
        re.compile(r"(?<!\.)run\([^)]*shell\s*=\s*True", re.DOTALL),    # unqualified run (if imported)
        re.compile(r"(?<!\.)call\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)check_output\([^)]*shell\s*=\s*True", re.DOTALL),
    ],
    # CWE-89: SQL Injection
    "CWE-89": [
        re.compile(r"\.execute\([^)]*(SELECT|INSERT|UPDATE|DELETE)", re.IGNORECASE | re.DOTALL),
        re.compile(r"\.execute\([^)]*\+[^)]*\)", re.DOTALL),       # string concatenation in query
        re.compile(r"\.execute\([^)]*\.format\(", re.DOTALL),      # use of format() in query
        re.compile(r"\.execute\([^)]*f[\'\"]", re.DOTALL),         # f-string in query
        re.compile(r"\.execute\([^)]*%[^)]*\)", re.DOTALL),        # %-formatting in query
    ],
    # CWE-79: Cross-Site Scripting (XSS)
    "CWE-79": [
        re.compile(r"mark_safe\(", re.DOTALL),
        re.compile(r"jinja2\.Environment\([^)]*autoescape\s*=\s*False", re.DOTALL),
        re.compile(r"autoescape\s*=\s*False", re.DOTALL),  # any occurrence of disabling autoescape
        re.compile(r"mako\.template", re.DOTALL),
        re.compile(r"mako\.Template", re.DOTALL),
    ],
    # CWE-94: Code Injection (Eval/Exec)
    "CWE-94": [
        re.compile(r"\bexec\s*\(", re.DOTALL),
        re.compile(r"\beval\s*\(", re.DOTALL),
    ],
    # CWE-259: Hardcoded Password
    "CWE-259": [
        re.compile(r"password\s*=\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
        re.compile(r"['\"]password['\"]\s*:\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
    ],
    # CWE-732: Insecure File Permissions
    "CWE-732": [
        re.compile(r"os\.chmod\([^,]+,\s*(?:0o777|777|0x1ff|511)\b", re.DOTALL),
    ],
    # CWE-605: Binding to All Interfaces
    "CWE-605": [
        re.compile(r"0\.0\.0\.0"),
    ],
    # CWE-295: Improper Certificate Validation
    "CWE-295": [
        re.compile(r"requests\.(?:get|post|put|delete|head|options)\([^)]*verify\s*=\s*False", re.DOTALL),
        re.compile(r"ssl\._create_unverified_context\(", re.DOTALL),
        re.compile(r"set_missing_host_key_policy\([^)]*AutoAddPolicy", re.DOTALL),
    ],
    # CWE-330: Use of Insufficiently Random Values
    "CWE-330": [
        re.compile(r"random\.(?:Random|random|randint|randrange|choice|choices|uniform|triangular|sample|getrandbits|randbytes)\(", re.DOTALL),
    ],
    # CWE-502: Deserialization of Untrusted Data
    "CWE-502": [
        re.compile(r"pickle\.load(s|s)?\(", re.DOTALL),
        re.compile(r"dill\.load(s|s)?\(", re.DOTALL),
        re.compile(r"shelve\.open\(", re.DOTALL),
        re.compile(r"marshal\.load(s|s)?\(", re.DOTALL),
        re.compile(r"jsonpickle\.decode\(", re.DOTALL),
    ],
    # CWE-20: Improper Input Validation (e.g., unsafe YAML load)
    "CWE-20": [
        re.compile(r"yaml\.load\(", re.DOTALL),
    ],
    # CWE-22: Path Traversal (unsafe file extraction)
    "CWE-22": [
        re.compile(r"tarfile\.extractall\(", re.DOTALL),
        re.compile(r"tarfile\.extract\(", re.DOTALL),
        re.compile(r"ZipFile\.extractall\(", re.DOTALL),
        re.compile(r"ZipFile\.extract\(", re.DOTALL),
        re.compile(r"extractall\(", re.DOTALL),  # covers zipfile objects
    ],
    # CWE-703: Improper Error Handling
    "CWE-703": [
        re.compile(r"except\s*:\s*pass", re.DOTALL),
        re.compile(r"except\s*:\s*continue", re.DOTALL),
        re.compile(r"\bassert\s+", re.DOTALL),
    ],
    # CWE-327: Use of Broken or Risky Cryptographic Algorithm
    "CWE-327": [
        re.compile(r"hashlib\.(?:md5|sha1)\(", re.DOTALL),
        re.compile(r"hashlib\.new\(\s*['\"](?:MD5|md5|SHA1|sha1)['\"]", re.DOTALL),
        re.compile(r"(?:\b|\.)?(?:ARC2|ARC4|Blowfish|DES|XOR|CAST5|IDEA|SEED|TripleDES)\.new\(", re.DOTALL),
        re.compile(r"ssl\.PROTOCOL_(?:SSLv2|SSLv3|TLSv1)\b"),
    ],
}

predicted_cwes = []
temp_file = "__bandit_temp_code.py"

for code in df['vulnerable_function_source']:
    code_str = "" if pd.isna(code) else str(code)
    if code_str.strip() == "":
        predicted_cwes.append("")  # empty snippet
        continue

    # Write code snippet to a temp file for Bandit analysis
    try:
        with open(temp_file, "w", encoding="utf-8") as f:
            f.write(code_str)
    except Exception as e:
        # If writing fails, skip Bandit and use regex
        bandit_cwe_list = []
    else:
        # Run Bandit security scanner on the snippet
        bandit_cwe_list = []
        try:
            result = subprocess.run(
                ["bandit", "-f", "json", "-q", temp_file],
                capture_output=True, text=True, timeout=10
            )
            # If Bandit ran successfully, parse JSON output
            if result.returncode == 0 and result.stdout:
                data = json.loads(result.stdout)
                if data.get("results"):
                    for issue in data["results"]:
                        cwe_info = issue.get("issue_cwe")
                        if cwe_info:
                            cwe_id = cwe_info.get("id")
                            if cwe_id:
                                bandit_cwe_list.append(f"CWE-{cwe_id}")
                # If Bandit reported errors and no results, treat as no CWE found
                if data.get("errors") and not bandit_cwe_list:
                    bandit_cwe_list = []
            else:
                # Non-zero return code or no output -> assume Bandit did not find issues or failed
                bandit_cwe_list = []
        except Exception:
            # If Bandit is not installed or execution failed, fallback to regex
            bandit_cwe_list = []

    # Remove duplicate CWEs from Bandit results
    bandit_cwe_list = sorted(set(bandit_cwe_list), key=lambda x: int(x.split('-')[1])) if bandit_cwe_list else []

    if bandit_cwe_list:
        # Use CWE IDs reported by Bandit
        cwe_ids = bandit_cwe_list
    else:
        # Fallback: use regex patterns to infer CWE IDs
        found_cwe_set = set()
        for cwe, patterns in regex_patterns.items():
            for pattern in patterns:
                if pattern.search(code_str):
                    found_cwe_set.add(cwe)
                    break  # one match is enough for this CWE
        cwe_ids = sorted(found_cwe_set, key=lambda x: int(x.split('-')[1])) if found_cwe_set else []

    # Format the list of CWE IDs as a comma-separated string (or empty)
    predicted_cwes.append(", ".join(cwe_ids) if cwe_ids else "")

# Cleanup temp file
if os.path.exists(temp_file):
    os.remove(temp_file)

# Add the results to DataFrame and save to output CSV
df['predicted_cwe_ids'] = predicted_cwes
try:
    df.to_csv(output_csv, index=False)
    print(f"Predicted CWE IDs saved to '{output_csv}'.")
except Exception as e:
    print(f"Error writing output CSV: {e}")


In [ ]:
df['predicted_cwe_ids'].value_counts()

In [ ]:
import pandas as pd

df = pd.read_csv('ground_truth_functions_for_airflow.csv')
df = df.dropna(subset=['vulnerable_function_source'])

df = df[df['vulnerable_function_source'].str.strip() != '']
print(df.shape)
df.info()

In [ ]:
df.to_csv('functions_for_airflow.csv',index= False,quoting =1)

In [ ]:

import re
import subprocess
import json
import os
import sys

# Check for pandas installation
try:
    import pandas as pd
except ImportError:
    print("Pandas is not installed. Install it using 'pip install pandas'.")
    sys.exit(1)

# Parse command-line arguments for input and output file paths

input_csv = "functions_for_airflow.csv"
output_csv = "functions_for_airflow_with_cwe.csv"

# Load the dataset
try:
    df = pd.read_csv(input_csv)
except Exception as e:
    print(f"Error reading CSV file: {e}")
    sys.exit(1)

if 'vulnerable_function_source' not in df.columns:
    print("Input CSV must contain a 'vulnerable_function_source' column.")
    sys.exit(1)

# Regex patterns mapping common vulnerability patterns to CWE IDs (fallback heuristic)
regex_patterns = {
    # CWE-78: OS Command Injection
    "CWE-78": [
        re.compile(r"os\.system\(", re.DOTALL),
        re.compile(r"os\.popen\(", re.DOTALL),
        re.compile(r"subprocess\.(?:run|Popen|call|check_output)\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)Popen\([^)]*shell\s*=\s*True", re.DOTALL),  # unqualified Popen (if imported)
        re.compile(r"(?<!\.)run\([^)]*shell\s*=\s*True", re.DOTALL),    # unqualified run (if imported)
        re.compile(r"(?<!\.)call\([^)]*shell\s*=\s*True", re.DOTALL),
        re.compile(r"(?<!\.)check_output\([^)]*shell\s*=\s*True", re.DOTALL),
    ],
    # CWE-89: SQL Injection
    "CWE-89": [
        re.compile(r"\.execute\([^)]*(SELECT|INSERT|UPDATE|DELETE)", re.IGNORECASE | re.DOTALL),
        re.compile(r"\.execute\([^)]*\+[^)]*\)", re.DOTALL),       # string concatenation in query
        re.compile(r"\.execute\([^)]*\.format\(", re.DOTALL),      # use of format() in query
        re.compile(r"\.execute\([^)]*f[\'\"]", re.DOTALL),         # f-string in query
        re.compile(r"\.execute\([^)]*%[^)]*\)", re.DOTALL),        # %-formatting in query
    ],
    # CWE-79: Cross-Site Scripting (XSS)
    "CWE-79": [
        re.compile(r"mark_safe\(", re.DOTALL),
        re.compile(r"jinja2\.Environment\([^)]*autoescape\s*=\s*False", re.DOTALL),
        re.compile(r"autoescape\s*=\s*False", re.DOTALL),  # any occurrence of disabling autoescape
        re.compile(r"mako\.template", re.DOTALL),
        re.compile(r"mako\.Template", re.DOTALL),
    ],
    # CWE-94: Code Injection (Eval/Exec)
    "CWE-94": [
        re.compile(r"\bexec\s*\(", re.DOTALL),
        re.compile(r"\beval\s*\(", re.DOTALL),
    ],
    # CWE-259: Hardcoded Password
    "CWE-259": [
        re.compile(r"password\s*=\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
        re.compile(r"['\"]password['\"]\s*:\s*['\"][^'\"]+['\"]", re.IGNORECASE | re.DOTALL),
    ],
    # CWE-732: Insecure File Permissions
    "CWE-732": [
        re.compile(r"os\.chmod\([^,]+,\s*(?:0o777|777|0x1ff|511)\b", re.DOTALL),
    ],
    # CWE-605: Binding to All Interfaces
    "CWE-605": [
        re.compile(r"0\.0\.0\.0"),
    ],
    # CWE-295: Improper Certificate Validation
    "CWE-295": [
        re.compile(r"requests\.(?:get|post|put|delete|head|options)\([^)]*verify\s*=\s*False", re.DOTALL),
        re.compile(r"ssl\._create_unverified_context\(", re.DOTALL),
        re.compile(r"set_missing_host_key_policy\([^)]*AutoAddPolicy", re.DOTALL),
    ],
    # CWE-330: Use of Insufficiently Random Values
    "CWE-330": [
        re.compile(r"random\.(?:Random|random|randint|randrange|choice|choices|uniform|triangular|sample|getrandbits|randbytes)\(", re.DOTALL),
    ],
    # CWE-502: Deserialization of Untrusted Data
    "CWE-502": [
        re.compile(r"pickle\.load(s|s)?\(", re.DOTALL),
        re.compile(r"dill\.load(s|s)?\(", re.DOTALL),
        re.compile(r"shelve\.open\(", re.DOTALL),
        re.compile(r"marshal\.load(s|s)?\(", re.DOTALL),
        re.compile(r"jsonpickle\.decode\(", re.DOTALL),
    ],
    # CWE-20: Improper Input Validation (e.g., unsafe YAML load)
    "CWE-20": [
        re.compile(r"yaml\.load\(", re.DOTALL),
    ],
    # CWE-22: Path Traversal (unsafe file extraction)
    "CWE-22": [
        re.compile(r"tarfile\.extractall\(", re.DOTALL),
        re.compile(r"tarfile\.extract\(", re.DOTALL),
        re.compile(r"ZipFile\.extractall\(", re.DOTALL),
        re.compile(r"ZipFile\.extract\(", re.DOTALL),
        re.compile(r"extractall\(", re.DOTALL),  # covers zipfile objects
    ],
    # CWE-703: Improper Error Handling
    "CWE-703": [
        re.compile(r"except\s*:\s*pass", re.DOTALL),
        re.compile(r"except\s*:\s*continue", re.DOTALL),
        re.compile(r"\bassert\s+", re.DOTALL),
    ],
    # CWE-327: Use of Broken or Risky Cryptographic Algorithm
    "CWE-327": [
        re.compile(r"hashlib\.(?:md5|sha1)\(", re.DOTALL),
        re.compile(r"hashlib\.new\(\s*['\"](?:MD5|md5|SHA1|sha1)['\"]", re.DOTALL),
        re.compile(r"(?:\b|\.)?(?:ARC2|ARC4|Blowfish|DES|XOR|CAST5|IDEA|SEED|TripleDES)\.new\(", re.DOTALL),
        re.compile(r"ssl\.PROTOCOL_(?:SSLv2|SSLv3|TLSv1)\b"),
    ],
}

predicted_cwes = []
temp_file = "__bandit_temp_code.py"

for code in df['vulnerable_function_source']:
    code_str = "" if pd.isna(code) else str(code)
    if code_str.strip() == "":
        predicted_cwes.append("")  # empty snippet
        continue

    # Write code snippet to a temp file for Bandit analysis
    try:
        with open(temp_file, "w", encoding="utf-8") as f:
            f.write(code_str)
    except Exception as e:
        # If writing fails, skip Bandit and use regex
        bandit_cwe_list = []
    else:
        # Run Bandit security scanner on the snippet
        bandit_cwe_list = []
        try:
            result = subprocess.run(
                ["bandit", "-f", "json", "-q", temp_file],
                capture_output=True, text=True, timeout=10
            )
            # If Bandit ran successfully, parse JSON output
            if result.returncode == 0 and result.stdout:
                data = json.loads(result.stdout)
                if data.get("results"):
                    for issue in data["results"]:
                        cwe_info = issue.get("issue_cwe")
                        if cwe_info:
                            cwe_id = cwe_info.get("id")
                            if cwe_id:
                                bandit_cwe_list.append(f"CWE-{cwe_id}")
                # If Bandit reported errors and no results, treat as no CWE found
                if data.get("errors") and not bandit_cwe_list:
                    bandit_cwe_list = []
            else:
                # Non-zero return code or no output -> assume Bandit did not find issues or failed
                bandit_cwe_list = []
        except Exception:
            # If Bandit is not installed or execution failed, fallback to regex
            bandit_cwe_list = []

    # Remove duplicate CWEs from Bandit results
    bandit_cwe_list = sorted(set(bandit_cwe_list), key=lambda x: int(x.split('-')[1])) if bandit_cwe_list else []

    if bandit_cwe_list:
        # Use CWE IDs reported by Bandit
        cwe_ids = bandit_cwe_list
    else:
        # Fallback: use regex patterns to infer CWE IDs
        found_cwe_set = set()
        for cwe, patterns in regex_patterns.items():
            for pattern in patterns:
                if pattern.search(code_str):
                    found_cwe_set.add(cwe)
                    break  # one match is enough for this CWE
        cwe_ids = sorted(found_cwe_set, key=lambda x: int(x.split('-')[1])) if found_cwe_set else []

    # Format the list of CWE IDs as a comma-separated string (or empty)
    predicted_cwes.append(", ".join(cwe_ids) if cwe_ids else "CWE-Unknown")

# Cleanup temp file
if os.path.exists(temp_file):
    os.remove(temp_file)

# Add the results to DataFrame and save to output CSV
df['predicted_cwe_ids'] = predicted_cwes
try:
    df.to_csv(output_csv, index=False)
    print(f"Predicted CWE IDs saved to '{output_csv}'.")
except Exception as e:
    print(f"Error writing output CSV: {e}")


In [ ]:
df['predicted_cwe_ids'].value_counts()

In [ ]:
p#!/usr/bin/env python3
"""
Bandit-Only CWE Mapping Script

Reads a CSV with a 'vulnerable_function_source' column, runs Bandit
on each snippet to extract CWE IDs, and writes a new CSV with
a 'predicted_cwe_ids' column.
"""

import os
import sys
import json
import tempfile
import subprocess
import pandas as pd

# --- CONFIGURATION ---
INPUT_CSV  = "functions_for_tornado.csv"
OUTPUT_CSV = "functions_for_tornado_with_cwe_bandit.csv"

# --- LOAD DATA ---
try:
    df = pd.read_csv(INPUT_CSV)
except Exception as e:
    print(f"[ERROR] Failed to read {INPUT_CSV}: {e}")
    sys.exit(1)

if 'vulnerable_function_source' not in df.columns:
    print("[ERROR] Input CSV must contain 'vulnerable_function_source' column.")
    sys.exit(1)

# --- PREPARE OUTPUT COLUMN ---
predicted = []

# --- PROCESS EACH SNIPPET ---
with tempfile.TemporaryDirectory(prefix="bandit_snips_") as tmpdir:
    for idx, code in df['vulnerable_function_source'].items():
        # Skip empty or NaN
        snippet = str(code) if pd.notna(code) else ""
        snippet = snippet.strip()
        if not snippet:
            predicted.append("CWE-Unknown")
            continue

        # Write snippet to a temp file
        snippet_path = os.path.join(tmpdir, f"snippet_{idx}.py")
        with open(snippet_path, "w", encoding="utf-8") as f:
            f.write(snippet)

        # Run Bandit against this one file
        try:
            proc = subprocess.run(
                ["bandit", "-f", "json", "-q", "--exit-zero", snippet_path],
                capture_output=True, text=True, timeout=30
            )
            data = json.loads(proc.stdout or "{}")
        except Exception as e:
            # Bandit failed—mark unknown
            predicted.append("CWE-Unknown")
            continue

        # Extract CWE IDs
        cwes = set()
        for issue in data.get("results", []):
            cwe_info = issue.get("issue_cwe", {})
            if isinstance(cwe_info, dict) and cwe_info.get("id"):
                cwes.add(f"CWE-{cwe_info['id']}")

        # Record result
        if cwes:
            # comma-separated if multiple
            predicted.append(", ".join(sorted(cwes)))
        else:
            predicted.append("CWE-Unknown")

# --- SAVE RESULTS ---
df['predicted_cwe_ids'] = predicted
try:
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"[INFO] Saved CWE mappings to {OUTPUT_CSV}")
except Exception as e:
    print(f"[ERROR] Failed to write {OUTPUT_CSV}: {e}")
    sys.exit(1)


In [ ]:
df['predicted_cwe_ids'].value_counts()